# Form Analyzer (TF pose sequences)

# Form Analyzer — TF temporal pose model (ST-GCN / 1D-CNN over 17-joint sequences)

Processes `training/process_form_data.py` output (`data/processed/forms/normalised.json`):
normalised 17-joint keypoint sequences + per-exercise quality labels
(`good` / `slight_misalignment` / `bad`). Matches the serving contract of
`app/form_analyzer_engine.py` (shoulder-width = 1, origin = hip midpoint).

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
from tf_utils import set_memory_growth
set_memory_growth()


In [ ]:
import json
from pathlib import Path
proc = Path('../data/processed/forms/normalised.json')
rows = json.loads(proc.read_text()) if proc.exists() else []
print('normalised sequences:', len(rows))
exercises = sorted({r['exercise'] for r in rows})
labels = sorted({r['label'] for r in rows})
print('exercises:', exercises, '| labels:', labels)

In [ ]:
import numpy as np
import tensorflow as tf

def pad_frames(seq, T=64):
    T = min(T, len(seq)); seq = seq[:T]
    out = np.zeros((64, 17, 3), dtype=np.float32)
    out[:len(seq)] = seq
    return out

X = np.stack([pad_frames(r['normalised']) for r in rows])
y = np.array([labels.index(r['label']) for r in rows])
ds = tf.data.Dataset.from_tensor_slices((X, tf.keras.utils.to_categorical(y, len(labels))))
ds = ds.batch(16).prefetch(tf.data.AUTOTUNE)

In [ ]:
import tensorflow as tf
inp = tf.keras.Input(shape=(64, 17, 3))
x = tf.keras.layers.Conv2D(32, (3,1), padding='same', activation='relu')(inp)
x = tf.keras.layers.Conv2D(64, (3,1), padding='same', activation='relu')(x)
x = tf.keras.layers.Reshape((64, 17*64))(x)  # temporal-axis pooling is a TODO
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
out = tf.keras.layers.Dense(len(labels), activation='softmax')(x)
m = tf.keras.Model(inp, out)
m.compile('adam', 'categorical_crossentropy', metrics=['accuracy'])
m.summary()

In [ ]:
# NOTE: replace the pooling above with an ST-GCN / temporal-Transformer
# (research lane) for real sequences; this is a working 1D-CNN baseline.
m.fit(ds, epochs=30)

In [ ]:
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
onnx = export_keras_onnx(m, Path('../models'), 'form_analyzer', '1.0.0')
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name':'form_analyzer','version':'1.0.0','artifact_path':str(q),
            'framework':'tensorflow','metrics':{'acc':0.0}})